<a href="https://colab.research.google.com/github/weagan/ARC-AGI-3/blob/main/arc_agi3_multigame_orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ARC-AGI-3 Fault-Tolerant Iterative Multi-Game Orchestrator
This notebook implements the production-grade, hardened **Multi-Game Duck Harness** with an integrated **Exponential Backoff Request Supervisor** [cite: 2]. It actively catches `ServerError 503 UNAVAILABLE` exception spikes and applies random jitter backoff to guarantee uninterrupted execution during peak traffic surges [cite: 2].

### 1. Multi-Platform Credential Manager
Safely extracts credentials across workspace infrastructure environments [cite: 2].

In [1]:
import os

def load_api_key(secret_name="GEMINI_API_KEY"):
    """Loads a secret securely from Google Colab or Kaggle secret managers [cite: 2]."""
    try:
        from google.colab import userdata
        return userdata.get(secret_name)
    except ImportError:
        pass

    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        return user_secrets.get_secret(secret_name)
    except ImportError:
        pass

    return os.environ.get(secret_name, "")

API_KEY = load_api_key()
print("Secret loaded successfully (masked for privacy) [cite: 2].")

Secret loaded successfully (masked for privacy) [cite: 2].


### 2. Multi-Game Environment Suite Definition
Establishes Game A (Agent Navigation Maze) and Game B (Global Matrix Symmetry Alignment Puzzles) [cite: 2].

In [2]:
import numpy as np

class AgentMovementGame:
    def __init__(self):
        self.control_layout = "AGENT_MOVEMENT_KEYS"
        self.grid = np.array([
            [1, 1, 1, 1, 1],
            [1, 2, 0, 0, 1],
            [1, 1, 1, 0, 1],
            [1, 3, 0, 0, 1],
            [1, 1, 1, 1, 1]
        ])
        self.player_pos = (1, 1)

    def get_state(self):
        return self.grid.copy()

    def step(self, direction):
        r, c = self.player_pos
        if direction == 'UP': r -= 1
        elif direction == 'DOWN': r += 1
        elif direction == 'LEFT': c -= 1
        elif direction == 'RIGHT': c += 1

        if self.grid[r, c] != 1:
            self.grid[self.player_pos] = 0
            if self.grid[r, c] == 3:
                print("[Game Engine A] Success: Goal achieved!")
            self.player_pos = (r, c)
            self.grid[r, c] = 2
        return self.get_state()

class GlobalOrchestrationGame:
    def __init__(self):
        self.control_layout = "GLOBAL_MATRIX_EDIT_KEYS"
        self.grid = np.array([
            [4, 0, 0, 0],
            [4, 0, 0, 0],
            [4, 0, 0, 0],
            [4, 0, 0, 4]
        ])

    def get_state(self):
        return self.grid.copy()

    def update_cell(self, row, col, value):
        if 0 <= row < 4 and 0 <= col < 4:
            self.grid[row, col] = value

    def click_confirm(self):
        if np.array_equal(self.grid[:, 0], self.grid[:, 3]):
            print("[Game Engine B] Success: Global Symmetry Criteria Validated!")
        else:
            print("[Game Engine B] Failed: Pattern does not meet criteria.")
        return self.get_state()

### 3. Reflective Harness Framework
Manages textual layout serialization and execution sandboxing [cite: 2].

In [3]:
import sys
from io import StringIO

class ReflectiveDuckHarness:
    def __init__(self):
        self.official_system_prompt = (
            "You are playing a game. Your goal is to win. Reply with the exact action you want to take. "
            "The final action in your reply will be executed next turn. Your entire reply will be carried to the next turn."
        )

    def serialize_grid(self, grid):
        return "\n".join([" ".join(map(str, row)) for row in grid])

    def assemble_prompt_with_feedback(self, grid, control_type, feedback_log=None):
        grid_str = self.serialize_grid(grid)
        feedback_block = ""
        if feedback_log:
            feedback_block = f"""=== CRITICAL EXECUTION FEEDBACK FROM PREVIOUS ATTEMPT ===\n{feedback_log}\n=========================================================\n"""
        return f"""=== ARC-AGI-3 CORE SYSTEM ENVIRONMENT ===
{self.official_system_prompt}

{feedback_block}=== CURRENT GAME WORKING INTERFACE ===
Grid Array:
{grid_str}

Available Control Protocol: {control_type}

=== TASKS ===
1. Analyze the grid layout and deduce the goal of the game using Core Knowledge priors.
2. Generate valid executable Python code using the layout's methods:
   - For AGENT_MOVEMENT_KEYS: `env.step('UP'/'DOWN'/'LEFT'/'RIGHT')` [cite: 2]
   - For GLOBAL_MATRIX_EDIT_KEYS: `env.update_cell(r, c, val)` and `env.click_confirm()` [cite: 2]

Return ONLY the executable Python script block inside markdown.
"""

    def execute_code_repl(self, code_str, env):
        old_stdout = sys.stdout
        redirected_output = StringIO()
        sys.stdout = redirected_output
        execution_scope = {"env": env, "harness": self, "__builtins__": __builtins__}
        try:
            exec(code_str, execution_scope, execution_scope)
            error = None
        except Exception as e:
            error = str(e)
        sys.stdout = old_stdout
        return redirected_output.getvalue(), error

### 4. Interactive Feedback Evaluation Pipeline (With Server Backoff Fix)
Wraps model calls in exponential delays to absorb 503 spike errors seamlessly [cite: 2].

In [4]:
from google import genai
from google.genai.errors import ServerError, APIError
import re
import time
import random

client = genai.Client(api_key=API_KEY)
harness = ReflectiveDuckHarness()

def call_model_with_backoff(client, model_id, prompt, max_api_retries=5):
    base_delay = 2.0
    for attempt in range(max_api_retries):
        try:
            return client.models.generate_content(model=model_id, contents=prompt)
        except (ServerError, APIError) as e:
            if attempt == max_api_retries - 1:
                raise e
            sleep_duration = (base_delay ** attempt) + random.uniform(0.1, 0.9)
            print(f"  [API Warning] Endpoint overloaded (503). Retrying in {sleep_duration:.2f}s...")
            time.sleep(sleep_duration)

games = [
    ("Game Type A (Agent Step Maze)", AgentMovementGame()),
    ("Game Type B (Global Matrix Orchestration)", GlobalOrchestrationGame())
]

for game_title, env in games:
    print(f"\n=======================================================")
    print(f"RUNNING ITERATIVE HARNESS LOOP FOR: {game_title}")
    print(f"=======================================================")

    feedback_tracker = None
    max_iterations = 3

    for attempt in range(max_iterations):
        print(f"\n--- [Attempt {attempt + 1}/{max_iterations}] Compiling Context Prompt ---")
        prompt = harness.assemble_prompt_with_feedback(env.get_state(), env.control_layout, feedback_tracker)

        if not API_KEY:
            if env.control_layout == "AGENT_MOVEMENT_KEYS":
                script_block = "env.step('RIGHT')\nenv.step('RIGHT')\nenv.step('DOWN')\nenv.step('DOWN')\nenv.step('LEFT')\nenv.step('LEFT')"
            else:
                if attempt == 0:
                    script_block = "env.update_cell(0,0,8)\nenv.click_confirm()"
                else:
                    script_block = "env.update_cell(0,3,4)\nenv.update_cell(1,3,4)\nenv.update_cell(2,3,4)\nenv.click_confirm()"
        else:
            response = call_model_with_backoff(client, 'gemini-2.5-flash', prompt)
            blocks = re.findall(r'```python\s*(.*?)\s*```', response.text, re.DOTALL)
            script_block = blocks[0] if blocks else response.text

        print(f"Captured Script Block:\n{script_block.strip()}")
        output, err = harness.execute_code_repl(script_block, env)

        print(f"Harness Run Result Output:\n{output.strip() if output else 'Empty Out.'}")
        # Fixed string literal matching using single quotes to avoid JSON syntax errors
        if 'Success' in output:
            print(">> Objective Completed Successfully!")
            break
        else:
            feedback_tracker = f"Stdout Log: {output.strip() if output else 'None'}. Runtime Exception: {err if err else 'None'}"

    print("\n--- Verified Final State Layout Array ---")
    print(harness.serialize_grid(env.get_state()))


RUNNING ITERATIVE HARNESS LOOP FOR: Game Type A (Agent Step Maze)

--- [Attempt 1/3] Compiling Context Prompt ---
Captured Script Block:
env.step('DOWN')
env.step('DOWN')
Harness Run Result Output:
Empty Out.

--- [Attempt 2/3] Compiling Context Prompt ---
Captured Script Block:
env.step('RIGHT')
env.step('RIGHT')
env.step('DOWN')
env.step('DOWN')
env.step('LEFT')
env.step('LEFT')
Harness Run Result Output:
[Game Engine A] Success: Goal achieved!
>> Objective Completed Successfully!

--- Verified Final State Layout Array ---
1 1 1 1 1
1 0 0 0 1
1 1 1 0 1
1 2 0 0 1
1 1 1 1 1

RUNNING ITERATIVE HARNESS LOOP FOR: Game Type B (Global Matrix Orchestration)

--- [Attempt 1/3] Compiling Context Prompt ---
Captured Script Block:
env.update_cell(3, 3, 0)
env.click_confirm()
Harness Run Result Output:
[Game Engine B] Failed: Pattern does not meet criteria.

--- [Attempt 2/3] Compiling Context Prompt ---
Captured Script Block:
# Task 1: Analyze the grid layout and deduce the goal of the game usi

# Mathematical Blueprint of ARC-AGI-3 Agent Frameworks

## Part 1: The Duck Harness Baseline

### 1. The Neural-Symbolic State Transition Function
At any given time step t, the environment produces a raw grid observation O_t ∈ G, where G denotes the set of grid configurations bounded by 64 × 64. The harness operationalizes this observation into a language-accessible programmatic state S_t via a deterministic execution function:

$$S_t = \text{REPL}(S_{t-1}, A_{t-1}, O_t)$$

where A_{t-1} represents the executable Python code action generated by the agent at the previous step, and REPL serves as the interpreter executing code inside a sandboxed state space.

### 2. Prioritized Experience Replay and Hashing Function
To avoid cyclic trajectories, unique configurations are tracked via a state hashing function H mapping a multidimensional grid representation into a unique scalar key k_t:

$$H: S_t \to k_t \in \mathbb{N}$$

The prioritized replay buffer B transitions over time by incorporating state-action-reward tuples, sampled using an optimization priority metric δ_t:

$$\mathcal{B}_{t} = \mathcal{B}_{t-1} \cup \{(S_t, A_t, R_t, S_{t+1})\}$$

$$P(\text{sampling } \tau) = \frac{\delta_\tau^\alpha}{\sum_i \delta_i^\alpha}$$

where δ scales explicitly with metrics tracking exploration novelty and goal inference velocity.

### 3. Context Retention & Message Eviction Policy
Let \mathcal{C}_t represent the context window history provided to the underlying large language model at step t. The total token length function L(\mathcal{C}_t) must obey a hard boundary constraint:

$$L(\mathcal{C}_t) \le T_{\text{max}}$$

where T_{\text{max}} is the context capability ceiling. To maintain this threshold, the Duck Harness establishes a dynamic message eviction scheme:

$$\mathcal{C}_t = \mathcal{C}_{t-1} \setminus \{ m_i \in \mathcal{C}_{t-1} \mid \text{Score}(m_i) < \gamma \} \cup \{ (S_t, O_t) \}$$

where \text{Score}(m_i) scores the long-term utility of past code execution trajectories.

### 4. Objective Action-Selection Mapping
The executable action script A_t is sampled from a code generation policy \pi parameterized by network parameters \theta:

$$A_t ~ \pi_{\theta}(A \mid \mathcal{C}_t)$$

---

## Part 2: SARBLOH \mathcal{O}(HW) Object Tracking Engine

### 1. Single-Pass Canonical Component Labeling
Let the input grid at step t be represented as a discrete matrix M \in \{0, \dots, C\}^{H \times W}, where C is the maximal color integer. Defining a neighborhood topology function N(x, y), the spatial engine maps equivalent labels across cell locations during a single array loop:

$$L(x, y) = \begin{cases} \text{new\_label}() & \text{if } \forall (x',y') \in N(x,y), M(x',y') \neq M(x,y) \\ \min_{(x',y')} \{ L(x', y') \} & \text{if } M(x',y') = M(x,y) \end{cases}$$

Resolving equivalence relationships through a path-compressed disjoint-set structure yields a complete spatial object inventory \mathbf{O}_t = \{o_1, o_2, \dots, o_k\}.

### 2. Geometric Coordinate Reduction & Invariant Briefing
Each isolated spatial component o_i \in \mathbf{O}_t is directly reduced to an invariant description bundle \psi(o_i) via a single-pass matrix traversal:

$$\psi(o_i) = \Big( \mathbf{c}_i, \mathbf{B}_i, \text{Color}_i, \text{Area}_i, \mathbf{\Sigma}_i \Big)$$

where the geometric attributes are computed via linear operations:
* Centroid Vector (\mathbf{c}_i): $$\mathbf{c}_i = \frac{1}{|o_i|} \sum_{(x,y) \in o_i} \begin{bmatrix} x \\ y \end{bmatrix}$$
* Bounding Box Boundaries (\mathbf{B}_i): $$\left[x_{\min}, x_{\max}\right] \times \left[y_{\min}, y_{\max}\right]$$
* Covariance Vector (\mathbf{\Sigma}_i): $$\mathbf{\Sigma}_i = \frac{1}{|o_i|} \sum_{(x,y) \in o_i} \left( \begin{bmatrix} x \\ y \end{bmatrix} - \mathbf{c}_i \right)\left( \begin{bmatrix} x \\ y \end{bmatrix} - \mathbf{c}_i \right)^T$$

### 3. Bipartite Temporal Tracking
To preserve object identities smoothly over sequential matrix frames without drift, an affinity metric calculates matching values across frames. Let \mathbf{O}_{t-1} and \mathbf{O}_t represent consecutive frame segments. A match weight matrix \mathbf{W} is constructed by comparing spatial properties:

$$W_{ij} = \alpha \cdot \text{IoU}(\mathbf{B}_i, \mathbf{B}_j) + \beta \cdot \delta(\text{Color}_i, \text{Color}_j) - \gamma \Vert{}\mathbf{c}_i - \mathbf{c}_j\Vert{}_2$$

The optimal sequential assignment mapping matrix \mathbf{A}^* is found by optimizing over the tracking weight space:

$$\mathbf{A}^* = \arg\max_{\mathbf{A}} \sum_{(i, j) \in \mathbf{A}} W_{ij}$$
